In [ ]:
import math
import numpy as np 
import pandas as pd
import xarray as xr

from datetime import datetime, timedelta

# libraries for interactive plots
import hvplot.pandas  # noqa
import holoviews as hv

from holoviews import opts

In [ ]:
# load internal scripts 
from pcr import helper, storm

In [ ]:
import importlib
importlib.reload(storm)

# Generating a Non-homogeneous Poisson Process
Based on Chapter 5.5 Generating a Nonhomogeneous Poisson Process in Simulation, Fifth Edition by Sheldon M. Ross (2013).  
DOI: https://doi.org/10.1016/C2011-0-04574-X

In the following chapter we are trying to implement the stochastic simulation described on the literature.

In [ ]:
# Define the lambda
lambda_mon = np.array([ 4.09756098,  6.73170732,  7.31707317,  9.95121951, 16.3902439 ,
       22.53658537, 19.90243902, 16.3902439 , 11.70731707, 11.41463415,
        6.73170732,  3.51219512])

lambda_0 = np.ceil(lambda_mon.max())

def lam(day_t, lams):
    date_t = date_start + np.timedelta64(int(day_t*24), 'h')
    
    if isinstance(date_t, np.datetime64):
        month = date_t.item().month
    elif isinstance(date_t, datetime):
        month = date_t.month

    return lams[month-1]

def lam_m(mon, lams):
    return lams[mon]

# Define the simulation time
# date_start = datetime(year=1979, month=1, day=1)
# date_end = datetime(year=2020, month=12, day=31)

date_start = np.datetime64('1979-01-01')
date_end = np.datetime64('2020-12-31')

t_days = (date_end-date_start).item().days

In [ ]:
lams=lambda_mon
T=t_days

lam_0 = lams.max()
t=0
N=0
s=[]

while t <= T:
    u1 = np.random.uniform()
    t = t-(math.log(u1)/lam_0) * 365
    if t <= T:
        u2 = np.random.uniform()
        if u2 <= lam(t, lams)/lam_0:
            N += 1
            s.append(t)

In [ ]:
out_df = pd.DataFrame({
    'time':[date_start + timedelta(days=day) for day in s], 
    'y': np.ones(len(s))
})

out_df['month'] = [s_dt.month for s_dt in out_df['time']]
count = out_df.groupby('month').count()

# get the measured wave storm
monthly_count_station = pd.read_csv('../data/output/monthly_count.csv')
monthly_count_station = monthly_count_station.rename(columns={'Unnamed: 0': 'month'}).set_index('month')

count = count.drop(columns='y').rename(columns={'time': 'thinning'})
count['measure'] = monthly_count_station['P23']
count['months'] = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

count.hvplot.bar(
    x='months', 
    y=['thinning', 'measure'], 
    ylabel='Monthly Storm Count',
).opts(multi_level=False)

Simulating 41-year wave storm as poisson process seems resulting a desirable results. Wrap the above procedure into a function. 

# Wrap into a Function
define a non-homogeneous poisson process function with parameter of (non-homogeneous) lambda and time of the simulation.

In [ ]:
def nonHomPois(lams, T):
    lam_0 = lams.max()
    t=0
    N=0
    s=[]

    while t <= T:
        u1 = np.random.uniform()
        t = t-(math.log(u1)/lam_0) * 365
        if t <= T:
            u2 = np.random.uniform()
            if u2 <= lam(t, lams)/lam_0:
                N += 1
                s.append(t)
    return(N, np.array(s))   


# Simulating Wave Storm Events
Experimenting on applying the written function to simulation
## One Realisation
Make 1 realisation of 41-years of wave storm

In [ ]:
# interpolated lambda 
inter_lambda_sl = np.array([30.43902439,  9.36585366,  1.17073171,  1.17073171,  1.17073171, 
                            0.87804878,  0.29268293,  1.46341463,  1.02439024,  0.58536585,
                            12.58536585, 35.12195122])

# wave storm count in 41 years 
count_sl = np.array([104.,  32.,   4.,  np.nan,   4.,   3.,   1.,   5.,  np.nan,   2.,  43., 120.])
count_sl_zero = np.nan_to_num(count_sl, nan=0, copy=True)

lambda_sl = count_sl / 41 * 12
np.nan_to_num(lambda_sl, nan=0, copy=False)

# Define the simulation time
date_start = datetime(year=1979, month=1, day=1)
date_end = datetime(year=2020, month=12, day=31)
# date_start = np.datetime64('1979-01-01')
# date_end = np.datetime64('2020-12-31')

# t_days = (date_end-date_start).item().days # for datetime64
t_days = (date_end-date_start).days # for datetime

n_storm, s = nonHomPois(lambda_sl, t_days)

In [ ]:
def monthly_bar(date_start, s, count_measure):
    srilanka_simulation = pd.DataFrame({
        'time':[date_start + timedelta(days=day) for day in s], 
        'y': np.ones(len(s))
    })

    count_sim_sl = pd.DataFrame(index=np.arange(1,13))

    srilanka_simulation['month'] = [s_dt.month for s_dt in srilanka_simulation['time']]
    count_sim_sl['thinning'] = srilanka_simulation[['month', 'y']].groupby('month').count()

    count_sim_sl['measure'] = count_measure
    count_sim_sl['months'] = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

    return count_sim_sl.hvplot.bar(
        x='months', 
        y=['thinning', 'measure'], 
        title='Storm Wave Count in Sri Lanka'
    ).opts(multi_level=False)


In [ ]:
monthly_bar(date_start, s, count_sl)

In [ ]:
srilanka_simulation = pd.DataFrame({
    'time':[date_start + timedelta(days=day) for day in s], 
    'y': np.ones(len(s))
})

count_sim_sl = pd.DataFrame(index=np.arange(1,13))

srilanka_simulation['month'] = [s_dt.month for s_dt in srilanka_simulation['time']]
count_sim_sl['thinning'] = srilanka_simulation[['month', 'y']].groupby('month').count()

count_sim_sl['measure'] = count_sl
count_sim_sl['months'] = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

count_sim_sl.hvplot.bar(
    x='months', 
    y=['thinning', 'measure'], 
    title='Storm Wave Count in Sri Lanka'
).opts(multi_level=False)

## Multiple realisations
do 10,000 of realisation of 41-years of wave storms

In [ ]:
# do many sims for 100 years 
numsim = 10000

res1 = []
for i in range(numsim) :
    res1.append(nonHomPois(lambda_sl, T))

In [ ]:
# post-processing: get the number of storm for 100-years
def count_bins(res1, numsim):
    bin = []
    for i in range(numsim): 
        bin.append(res1[i][0])

    p = pd.DataFrame(bin).hvplot.hist(
        bins=20,
        title='Distribution of Wave Storm Count in 41-years simulation (num of sim:10,000)', 
        label='Simulation'
    ).opts(
        show_legend=True
    )

    # height = p.dimension_values(1).max() *

    # spikes = hv.Spikes(([count_sl_zero.sum()], [height]), vdims='height', label='tes').opts(color='red', line_width=2)

    # p * spikes

    return (p * hv.VLine(count_sl_zero.sum(), label='Measured Wave storm 1979-2020')).opts(
        opts.VLine(color='red', show_legend=True))


In [ ]:
count_bins(res1, numsim)


In [ ]:
# post-processing: compare percentage of frequency of wave storm with measured
def freq_bins(res1, numsim, measure_count_df):
    count_sim = pd.DataFrame(
        {'count': np.zeros(12)},
        index=np.arange(1,13),
    )

    for i in range(numsim):
        out_sl = pd.DataFrame({
            'time':[date_start + timedelta(days=day) for day in res1[i][1]], 
        })

        out_sl['month'] = [s_dt.month for s_dt in out_sl['time']]

        count_sim['addition'] = out_sl.groupby('month').count()['time']
        count_sim['addition'] = count_sim['addition'].fillna(0)
        count_sim['count'] = count_sim['count'] + count_sim['addition']

    perc_sim_sl = pd.DataFrame(measure_count_df['measure'] / measure_count_df['measure'].sum())
    perc_sim_sl['simulation'] = count_sim['count'] / count_sim['count'].sum()

    perc_sim_sl['months'] = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

    return perc_sim_sl.hvplot.bar(
        x='months', 
        y=['measure', 'simulation'], 
        title='Percentage Frequency of Wave Storm in 100-years Simulation vs 41-years Data\n(num of sim: 10,000)'
    ).opts(multi_level=False)

In [ ]:
count_sl

In [ ]:
freq_bins(res1, numsim, measure_count_df=pd.DataFrame({
    'measure': count_sl,
}, 
    index=np.arange(1,13)
))


In [ ]:
def avg_count_bins(res1, numsim, measure_count_df):
    count_sim = pd.DataFrame(
        {'count': np.zeros(12)},
        index=np.arange(1,13),
    )

    for i in range(numsim):
        out_sl = pd.DataFrame({
            'time':[date_start + timedelta(days=day) for day in res1[i][1]], 
        })

        out_sl['month'] = [s_dt.month for s_dt in out_sl['time']]

        count_sim['addition'] = out_sl.groupby('month').count()['time']
        count_sim['addition'] = count_sim['addition'].fillna(0)
        count_sim['count'] = count_sim['count'] + count_sim['addition']

    perc_sim_sl = pd.DataFrame({
        'measure_count': measure_count_df['measure'], 
        'measure_percentage': measure_count_df['measure'] / measure_count_df['measure'].sum()
    })
    perc_sim_sl['simulation_percentage'] = count_sim['count'] / count_sim['count'].sum()
    perc_sim_sl['simulation_avg_count'] = count_sim['count'] / numsim

    perc_sim_sl['months'] = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

    return perc_sim_sl.hvplot.bar(
        x='months', 
        y=['measure_count', 'simulation_avg_count'], 
        title='Average of Wave Storm Count in 41-years Simulation vs Data\n(num of sim: 10,000)'
    ).opts(multi_level=False)


avg_count_bins(res1, numsim, pd.DataFrame({
    'measure': count_sl,
}, 
    index=np.arange(1,13)
))

# Simulating full process of wave storm 

In [ ]:
# test out the function using data from data/wave_srilanka.csv
# import wave data
data_path = '../data/ERA5/ts/sri_lanka_1979_2020.nc'
ds = xr.open_dataset(data_path)
era5_ds = ds.sel(valid_time=slice('1979-01-01', '2020-12-31'))

# get hs, dir, tp, time from dataframe
hs = era5_ds['swh'].values
dir = era5_ds['mwd'].values
tp = era5_ds['mwp'].values
time = helper.datetime64_to_datenum(pd.Series(era5_ds['valid_time'].values))

ts_hs = 95
ts_dur = 12.0

# detect storm 
storms, storms_ts = storm.detect(hs, dir, tp, time, ts_hs, ts_dur)

# fit storm and gap 
fitted_storms = storm.fit_storm(storms)
fitted_lambda = storm.fit_lambda(storms, fillna='zeros')

# generate storm sample
storms_sample = storm.generate(
    fitted_storm=fitted_storms, 
    sampling_size=1000, 
    oversample=0.1, 
    max_dur=np.max(storms.duration))

In [ ]:
# 41-years simulation (1979-2020)
sim_start = datetime(year=1979, month=1, day=1)
sim_end = datetime(year=2020, month=12, day=31)

T = (sim_end - sim_start).days

lams = fitted_lambda

def simulate_gap(lams, T, storms_sample):
    lam_0 = lams.max()+0.001
    t=0
    N=0
    s=[]

    while t <= T:
        u1 = np.random.uniform()
        t = t-(math.log(u1)/lam_0) * 365
        if t <= T:
            u2 = np.random.uniform()
            if u2 <= lam(t, lams)/lam_0:
                N += 1
                s.append(t)
                # add storm duration 
                t += storms_sample['duration'].iloc[N-1] / 24 

    return(N, np.array(s))

In [ ]:
n_one, s_one = simulate_gap(fitted_lambda, T, storms_sample)

# compare the simulation to measurement 
measure_count = storm.monthly_count(storms)['count'].values
monthly_bar(sim_start, s_one, measure_count)

In [ ]:
# do many sims for 100 years 
numsim = 10000

res_dur= []
for i in range(numsim) :
    res_dur.append(simulate_gap(lambda_sl, T, storms_sample))

In [ ]:
count_bins(res_dur, numsim)

In [ ]:
measure_count_df = pd.DataFrame(
    {'measure': measure_count}, 
    index=np.arange(1,13)
)
freq_bins(res_dur, numsim, measure_count_df)

In [ ]:
# post-processing: compare percentage of frequency of wave storm with measured
avg_count_bins(res_dur, numsim, measure_count_df)

In [ ]:
res_monthly = []

monthly = pd.DataFrame(
    index=np.arange(1,13),
)

for i in range(numsim): 

    out_sl = pd.DataFrame({
        'time':[date_start + timedelta(days=day) for day in res_dur[i][1]], 
    })

    out_sl['month'] = [s_dt.month for s_dt in out_sl['time']]

    monthly['month_count'] = out_sl.groupby('month').count()['time']
    monthly['month_count'] = monthly['month_count'].fillna(0)

    res_monthly.append(monthly['month_count'].values)

In [ ]:
monthly_df = pd.DataFrame(
    res_monthly, 
    columns=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
)

counted = monthly_df.stack().reset_index().drop('level_0', axis=1).rename(columns={'level_1': 'Month', 0: 'Count'})

In [ ]:
# counted.hvplot.scatter(
#     y='Count', 
#     x='Month', 
#     color='red',
#     marker='x', 
#     # size=8, 
#     alpha=0.01, 
#     width=600, 
#     height=400, 
#     grid=True, 
#     ylabel='Storm counts (events/41-years)'
# ) *\
pd.DataFrame({
    'measure': np.nan_to_num(count_sl, nan=0, copy=False), 
    'Months': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
}).hvplot.scatter(
    y='measure', 
    x='Months', 
    color='blue',
    size=20,
    fill_color='blue',
) *\
counted.hvplot.violin(
    y='Count', 
    by='Month', 
    violin_fill_alpha=0
)

# Fitting in Storm Duration 
Simulating arrival time of the storm using NHPP 

# Recap
Thinning method to simulate the "arrival" of storm has some potential. It might need to check a couple of things: 
1. The wave storm count seems shifted then the measured to the right -> have to check with storm duration 
    - if we add the storm duration on the accept and reject it skews the histogram far to the left 
    - let the NHPP be used for simulating the inter-arrival time of the storm 
2. clustering might happen -> we have to check 
    - This have to be checked: if the inter-arrival to short and it does not fit the storm duration. 


To-do: 
- incorporate with the rest of the routine: write the code in scripts 
- tambah duration 